# Agentic RAG Pipeline Foundation

Make sure you have your `.env` file set up with `GROQ_API_KEY=your_key` in the same directory as this notebook!

In [3]:
import os
from dotenv import load_dotenv


load_dotenv() 

api_key = os.environ.get("GROQ_API_KEY")


### Step 1: Collect / Create Documents

I'm going to create a few sample text files right here in a `documents` folder. Feel free to replace these with your actual resume, notes, or project READMEs!

In [4]:
import os

os.makedirs('documents', exist_ok=True)

docs = {
    "resume.txt": "My name is bikrant. I am a software engineer with 2 years of experience in Python, AI, and distributed systems. I love building agentic workflows.",
    "course_notes.txt": "Machine learning notes: RAG stands for Retrieval-Augmented Generation. It helps reduce hallucinations by providing ground-truth context to the LLM.Components of RAG.The main components of RAG are:External Knowledge Source: Stores domain specific or general information like documents, APIs or databases.Text Chunking and Preprocessing: Breaks large text into smaller, manageable chunks and cleans it for consistency.Embedding Model: Converts text into numerical vectors that capture semantic meaning.Vector Database: Stores embeddings and enables similarity search for fast information retrieval.Query Encoder: Transforms the user’s query into a vector for comparison with stored embeddings.Retriever: Finds and returns the most relevant chunks from the database based on query similarity.Prompt Augmentation Layer: Combines retrieved chunks with the user’s query to provide context to the LLM.LLM (Generator): Generates a grounded response using both the query and retrieved knowledge.Updater (Optional): Regularly refreshes and re-embeds data to keep the knowledge base up to date.Working of RAGThe system first searches external sources for relevant information based on the user’s query instead of relying only on existing training data.How-Rag-works-Training Creating External Data: External data from APIs, databases or documents is chunked, converted into embeddings and stored in a vector database to build a knowledge library.Retrieving Relevant Information: User queries are converted into vectors and matched against stored embeddings to fetch the most relevant data ensuring accurate responses.Augmenting the LLM Prompt: Retrieved content is added to the user’s query giving the LLM extra context to work with.Answer Generation: LLM uses both the query and retrieved data to generate a factually accurate, context aware response.Keeping Data Updated: External data and embeddings are refreshed regularly in real time or scheduled so the system always retrieves latest information.",
    "project_readme.txt": "Project Alpha: A local chromadb setup. It uses sentence-transformers for embedding and Groq for blazing fast inference.",
    "diary.txt": "Today I learned how to use chunking. Overlapping chunks make sure we don't cut important context in half!",
    "ideas.txt": "Startup idea: An AI that automatically writes your code based on jupyter notebook structures. Oh wait, that already exists."
}

for filename, content in docs.items():
    with open(os.path.join('documents', filename), 'w', encoding='utf-8') as f:
        f.write(content)
        
print("Created 5 sample documents in the 'documents' folder.")

Created 5 sample documents in the 'documents' folder.


### Step 2: Read and Chunk the Documents

We need to read these files and split them into overlapping chunks. I'll use `tiktoken` to count tokens roughly, aiming for ~300-500 tokens with a ~50 token overlap. Since our docs are short, this will mostly just keep them intact, but the logic scales to large PDFs.

In [8]:
import glob
import tiktoken

# let's use the standard cl100k_base tokenizer 
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_text(text, chunk_size=300, overlap=50):
    tokens = tokenizer.encode(text)
    chunks = []
    i = 0
    while i < len(tokens):
        chunk_tokens = tokens[i : i + chunk_size]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)
        i += chunk_size - overlap
    return chunks

all_chunks = []
chunk_metadata = []

for filepath in glob.glob('documents/*.txt'):
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
        
    chunks = chunk_text(text, chunk_size=300, overlap=50)
    for idx, c in enumerate(chunks):
        all_chunks.append(c)
        chunk_metadata.append({"source": filepath, "chunk_id": idx})

print(f"Created {len(all_chunks)} total chunks from our documents.")

Created 19 total chunks from our documents.


### Step 3: Embed and Store in ChromaDB

Now we embed the chunks using `sentence-transformers` (`all-MiniLM-L6-v2`) and store them in a local ChromaDB instance. Chroma will run in embedded mode right here, no background server needed!

In [9]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize local ChromaDB (it saves to a folder named 'chroma_db')
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# I use sentence-transformers all-MiniLM-L6-v2 as requested. 
# Chroma has a built-in wrapper for it!
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Create or get the collection
collection = chroma_client.get_or_create_collection(
    name="my_rag_collection",
    embedding_function=emb_fn
)

# Add chunks to DB
ids = [f"chunk_{i}" for i in range(len(all_chunks))]
collection.upsert(
    documents=all_chunks,
    metadatas=chunk_metadata,
    ids=ids
)

print("Successfully embedded and stored chunks in ChromaDB!")

Successfully embedded and stored chunks in ChromaDB!


### Step 4: Query and Generate Answer

Time for the fun part. We'll write a `query(question)` function that grabs the top-4 chunks, prints them so we can verify retrieval (super important for debugging!), and then asks our Groq LLM to answer the question using that context.

You mentioned using `gpt-oss-120b` on Groq, so I've set that as the model string!

In [10]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load API key from our .env file
load_dotenv()

groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def query(question):
    print(f"\n--- QUESTION: {question} ---")
    
    # 1. Retrieve top 4 most similar chunks
    results = collection.query(
        query_texts=[question],
        n_results=4
    )
    
    retrieved_chunks = results['documents'][0]
    sources = results['metadatas'][0]
    
    print("\n--- RETRIEVED CONTEXT ---")
    for idx, (chunk, meta) in enumerate(zip(retrieved_chunks, sources)):
        print(f"\n[Source: {meta['source']}]")
        print(chunk)
    print("\n---------------------------")
    
    # 2. Construct the prompt with the retrieved context
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""Use the following context to answer the question.\n
    If the answer is not in the context, say "I don't know based on the provided context."\n
    \n
    Context:\n
    {context}\n
    \n
    Question:\n
    {question}\n
    """
    
    # 3. Call Groq API
    
    response = groq_client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant answering questions based on the provided documents."},
            {"role": "user", "content": prompt}
        ],
        model="openai/gpt-oss-120b", 
        temperature=0.2
    )
    
    answer = response.choices[0].message.content
    print("\n--- AI ANSWER ---")
    print(answer)
    return answer

# Let's test it out!
query("What does RAG ?")


--- QUESTION: What does RAG ? ---

--- RETRIEVED CONTEXT ---

[Source: documents\course_notes.txt]
Machine learning notes: RAG stands for Retrieval-Augmented Generation. It helps reduce hallucinations by providing ground-truth context to the LLM.Components of RAG.The main components of RAG are:External Knowledge Source: Stores domain specific or general information like documents, APIs or databases.Text Chunking and Preprocessing: Breaks large text into smaller, manageable chunks and cleans it for consistency.Embedding Model: Converts text into numerical vectors that capture semantic meaning.Vector Database: Stores embeddings and enables similarity search for fast information retrieval.Query Encoder: Transforms the user’s query into a vector for comparison with stored embeddings.Retriever: Finds and returns the most relevant chunks from the database based on query similarity.Prompt Augmentation Layer: Combines retrieved chunks with the user’s query to provide context to the LLM.LLM (G

'RAG stands for **Retrieval‑Augmented Generation**.'